In [1]:
import pandas as pd
import requests
import os

# 1. Tạo thư mục lưu cache
os.makedirs("data/raw", exist_ok=True)
os.makedirs("data/processed", exist_ok=True)

# 2. Định nghĩa đủ 3 URL API SDMX chính thức theo Data Contract của nhóm
url_gdp_hours = "https://sdmx.oecd.org/public/rest/data/OECD.SDD.TPS,DSD_PDB@DF_PDB,2.0/DEU+USA+KOR+JPN.A.GDPHRS._T.USD_PPP_H.LR.N..?startPeriod=2005&endPeriod=2024&dimensionAtObservation=AllDimensions&format=csvfilewithlabels"
url_hours_worked = "https://sdmx.oecd.org/public/rest/data/OECD.ELS.SAE,DSD_HW@DF_AVG_ANN_HRS_WKD,1.0/DEU+USA+KOR+JPN........_T....?startPeriod=2005&endPeriod=2024&dimensionAtObservation=AllDimensions&format=csvfilewithlabels"
url_wages = "https://sdmx.oecd.org/public/rest/data/OECD.ELS.SAE,DSD_EARNINGS@AV_AN_WAGE,1.0/DEU+USA+KOR+JPN..USD_PPP....?startPeriod=2005&format=csvfilewithlabels"

def fetch_oecd_indicator(url, name):
    print(f"Đang gọi API tải dữ liệu cho chỉ tiêu: {name}...")
    response = requests.get(url)
    if response.status_code == 200:
        file_path = f"data/raw/{name}_raw.csv"
        with open(file_path, "wb") as f:
            f.write(response.content)
        print(f"✔ Tải thành công {name}!")
        return pd.read_csv(file_path)
    else:
        print(f"❌ Lỗi tải {name}: Mã trạng thái {response.status_code}")
        return None

# Tải 3 chỉ tiêu
df_gdp_raw = fetch_oecd_indicator(url_gdp_hours, "gdp_per_hour")
df_hours_raw = fetch_oecd_indicator(url_hours_worked, "annual_hours")
df_wages_raw = fetch_oecd_indicator(url_wages, "annual_wages")

# Hàm làm sạch và chuẩn hóa chung
def clean_dataframe(df, indicator_code, unit_name):
    if df is None or df.empty:
        return pd.DataFrame()
    cols = df.columns
    col_country_code = 'REF_AREA' if 'REF_AREA' in cols else [c for c in cols if 'AREA' in c][0]
    col_country_name = 'REF_AREA_ENT' if 'REF_AREA_ENT' in cols else col_country_code
    col_time = 'TIME_PERIOD' if 'TIME_PERIOD' in cols else [c for c in cols if 'TIME' in c or 'PERIOD' in c][0]
    col_value = 'OBS_VALUE' if 'OBS_VALUE' in cols else [c for c in cols if 'VALUE' in c][0]
    
    clean_df = df.dropna(subset=[col_value, col_time]).copy()
    
    result = pd.DataFrame()
    result['Country_Code'] = clean_df[col_country_code]
    result['Country'] = clean_df[col_country_name]
    result['Year'] = pd.to_numeric(clean_df[col_time], errors='coerce').astype(int)
    result['Indicator_Code'] = indicator_code
    result['Value'] = pd.to_numeric(clean_df[col_value], errors='coerce')
    result['Unit'] = unit_name
    return result.dropna(subset=['Value']).drop_duplicates()

# Xử lý làm sạch cho cả 3 chỉ tiêu
df_gdp_clean = clean_dataframe(df_gdp_raw, "GDPHRS", "USD_PPP_H")
df_hours_clean = clean_dataframe(df_hours_raw, "AVG_ANN_HRS", "Hours/worker")
df_wages_clean = clean_dataframe(df_wages_raw, "AV_AN_WAGE", "USD_PPP (Constant)")

# Gộp chung lại thành 1 file master duy nhất
final_master_df = pd.concat([df_gdp_clean, df_hours_clean, df_wages_clean], ignore_index=True)

# Xuất file bàn giao cho Team 3
output_path = "data/processed/oecd_productivity_clean.csv"
final_master_df.to_csv(output_path, index=False)

print(f"✔ ĐÃ CẬP NHẬT XONG! File sạch tổng hợp đã sẵn sàng tại: {output_path}")
print("Các mã Indicator có trong file hiện tại:", final_master_df['Indicator_Code'].unique())

Đang gọi API tải dữ liệu cho chỉ tiêu: gdp_per_hour...
✔ Tải thành công gdp_per_hour!
Đang gọi API tải dữ liệu cho chỉ tiêu: annual_hours...
✔ Tải thành công annual_hours!
Đang gọi API tải dữ liệu cho chỉ tiêu: annual_wages...
✔ Tải thành công annual_wages!
✔ ĐÃ CẬP NHẬT XONG! File sạch tổng hợp đã sẵn sàng tại: data/processed/oecd_productivity_clean.csv
Các mã Indicator có trong file hiện tại: <StringArray>
['GDPHRS', 'AVG_ANN_HRS', 'AV_AN_WAGE']
Length: 3, dtype: str


In [2]:
print("Đang tiến hành làm sạch và chuẩn hóa dữ liệu theo Data Contract...")

# Hàm chuẩn hóa chung cho từng DataFrame thô
def clean_dataframe(df, indicator_code, unit_name):
    if df is None or df.empty:
        return pd.DataFrame()
    
    # Tìm các cột tương ứng (thường OECD trả về REF_AREA, REF_AREA_ENT, TIME_PERIOD, OBS_VALUE)
    cols = df.columns
    col_country_code = 'REF_AREA' if 'REF_AREA' in cols else [c for c in cols if 'AREA' in c][0]
    col_country_name = 'REF_AREA_ENT' if 'REF_AREA_ENT' in cols else col_country_code
    col_time = 'TIME_PERIOD' if 'TIME_PERIOD' in cols else [c for c in cols if 'TIME' in c or 'PERIOD' in c][0]
    col_value = 'OBS_VALUE' if 'OBS_VALUE' in cols else [c for c in cols if 'VALUE' in c][0]
    
    clean_df = df.dropna(subset=[col_value, col_time]).copy()
    
    # Tạo cấu trúc chuẩn 6 cột theo Data Contract đã ký với Sub-team 3:
    # ['Country', 'Country_Code', 'Year', 'Indicator_Code', 'Value', 'Unit']
    result = pd.DataFrame()
    result['Country_Code'] = clean_df[col_country_code]
    result['Country'] = clean_df[col_country_name]
    result['Year'] = pd.to_numeric(clean_df[col_time], errors='coerce').astype(int)
    result['Indicator_Code'] = indicator_code
    result['Value'] = pd.to_numeric(clean_df[col_value], errors='coerce')
    result['Unit'] = unit_name
    
    return result.dropna(subset=['Value']).drop_duplicates()

# Xử lý từng chỉ tiêu dựa trên yêu cầu của Team 1
df_gdp_clean = clean_dataframe(df_gdp_raw, "GDPHRS", "USD_PPP_H")
df_hours_clean = clean_dataframe(df_hours_raw, "AVG_ANN_HRS", "Hours/worker")
df_wages_clean = clean_dataframe(df_wages_raw, "AV_AN_WAGE", "USD_PPP (Constant)")

# Hợp nhất (Concatenate) 3 chỉ tiêu thành một bảng dữ liệu thống nhất duy nhất
final_master_df = pd.concat([df_gdp_clean, df_hours_clean, df_wages_clean], ignore_index=True)

# Lưu file sạch vào thư mục processed để bàn giao cho Sub-team 3 (Dashboard)
output_path = "data/processed/oecd_productivity_clean.csv"
final_master_df.to_csv(output_path, index=False)

print(f"✔ HOÀN TẤT PIPELINE! Đã xuất file dữ liệu sạch tổng hợp tại: {output_path}")
display(final_master_df.head(10))

Đang tiến hành làm sạch và chuẩn hóa dữ liệu theo Data Contract...
✔ HOÀN TẤT PIPELINE! Đã xuất file dữ liệu sạch tổng hợp tại: data/processed/oecd_productivity_clean.csv


,Country_Code,Country,Year,Indicator_Code,Value,Unit
0,KOR,KOR,2020,GDPHRS,48.508713,USD_PPP_H
1,USA,USA,2007,GDPHRS,66.143589,USD_PPP_H
2,DEU,DEU,2019,GDPHRS,81.947738,USD_PPP_H
3,DEU,DEU,2020,GDPHRS,82.652236,USD_PPP_H
4,DEU,DEU,2021,GDPHRS,83.972784,USD_PPP_H
5,DEU,DEU,2018,GDPHRS,81.386192,USD_PPP_H
6,USA,USA,2008,GDPHRS,67.056967,USD_PPP_H
7,USA,USA,2009,GDPHRS,69.500466,USD_PPP_H
8,USA,USA,2010,GDPHRS,71.190611,USD_PPP_H
9,USA,USA,2011,GDPHRS,71.110020,USD_PPP_H
